# Module 6: Memory and Multi-Step Context
### Hands-on: build a multi-turn support assistant for "ShopEasy" (an online store)

**What you will learn**
1. Why LLMs are stateless (they forget everything between calls)
2. Conversation-buffer vs window vs summary memory
3. Token budgeting and context limits
4. Passing context across steps (agent → tool → agent)
5. **Lab:** add short-term memory so the agent remembers the whole session
6. Session (short-term) vs longer-term state

**Our story:** customer *Ravi* chats with the ShopEasy support bot about his order **1234**.

## Setup

In [ ]:
%pip install -qU langchain langchain-openai

In [ ]:
# API key: use Colab Secrets (key icon -> OPENAI_API_KEY), or type it when asked.
# Never hard-code keys in a notebook you share.
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
# The model (the brain)
from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-4o-mini"   # cheap, fast and good at using tools; swap in any OpenAI chat model you have access to

model = ChatOpenAI(model=MODEL_NAME, temperature=0, max_tokens=500)

print(model.invoke("Say hello in 5 words").content)

---
## Part 1: Why LLMs are stateless
An LLM is like a **support agent with no memory**: every call starts from zero.
It only knows what you send **in that one request**.

In [ ]:
# Call 1: we tell the model our name and order number
reply1 = model.invoke("Hi, my name is Ravi and my order number is 1234.")
print("Bot:", reply1.content)

In [ ]:
# Call 2: a new, separate call. Does it remember?
reply2 = model.invoke("What is my order number?")
print("Bot:", reply2.content)

**It doesn't know.** Each `invoke()` is independent, so the model never saw call 1.

> **Key idea:** "Memory" is not inside the model. **We** create memory by sending the earlier conversation along with each new message.

---
## Part 2: Conversation-buffer memory (keep everything)
The simplest memory: keep a **list of all messages** and send the whole list every time.

| Message type | Who | Example |
|---|---|---|
| `SystemMessage` | Instructions for the bot | "You are a ShopEasy support assistant" |
| `HumanMessage` | The user | "Where is my order?" |
| `AIMessage` | The bot's reply | "Your order is shipped" |
| `ToolMessage` | Result from a tool | "Status: Shipped" |

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

SYSTEM_TEXT = "You are a friendly support assistant for ShopEasy, an online store. Keep answers short (1-2 sentences)."

history = [SystemMessage(SYSTEM_TEXT)]      # this list IS the memory

def chat_buffer(user_text):
    history.append(HumanMessage(user_text))  # 1. add the user's message
    reply = model.invoke(history)            # 2. send the WHOLE history
    history.append(reply)                    # 3. add the bot's reply
    print("User:", user_text)
    print("Bot :", reply.content, "\n")

chat_buffer("Hi, my name is Ravi and my order number is 1234.")
chat_buffer("What is my order number?")
chat_buffer("And what is my name?")

In [ ]:
# Look inside the memory
for m in history:
    print(f"{m.type:>6}: {m.content[:70]}")

Now it remembers. But there's a catch: **the history keeps growing**, and every message is re-sent on every turn.
Let's measure it in **tokens** (roughly 1 token ≈ 4 characters of English).

In [ ]:
from langchain_core.messages.utils import count_tokens_approximately

print("Tokens sent right now:", count_tokens_approximately(history))

more_questions = [
    "Can I change the delivery address?",
    "Do you deliver on Sundays?",
    "What is your return policy?",
    "How do I get a refund?",
]
for q in more_questions:
    chat_buffer(q)
    print(f"   -> memory now holds {len(history)} messages, ~{count_tokens_approximately(history)} tokens\n")

Every turn costs more tokens, so it gets **slower, costlier**, and eventually hits the model's **context limit**.
That's why we need smarter memory.

---
## Part 3: Window memory and summary memory

### 3a. Window memory: keep only the last *k* turns
Cheap and simple, but **old facts are forgotten**.

In [ ]:
full_history = [SystemMessage(SYSTEM_TEXT)]

def chat_window(user_text, k=1):
    """Store everything, but SEND only the system message + last k turns + the new message."""
    full_history.append(HumanMessage(user_text))
    to_send = [full_history[0]] + full_history[1:][-(2 * k + 1):]
    reply = model.invoke(to_send)
    full_history.append(reply)
    print("User:", user_text)
    print(f"Bot : {reply.content}   (sent {len(to_send)} messages)\n")

chat_window("Hi, my name is Ravi.")
chat_window("My order number is 1234.")
chat_window("Do you deliver on Sundays?")
chat_window("What is my name?")      # the name is outside the window, so it's forgotten

### 3b. Summary memory: summarize old turns, keep recent ones word for word
Old messages are squeezed into a short **summary**, so key facts survive while tokens stay low.

In [ ]:
summary = ""        # short summary of older conversation
recent = []         # last few messages, kept exactly

def update_summary(old_summary, old_messages):
    convo = "\n".join(f"{m.type}: {m.content}" for m in old_messages)
    prompt = (
        "You maintain a summary of a customer support chat.\n"
        "Keep key facts: customer name, order numbers, problems, promises made. Max 3 short sentences.\n\n"
        f"Current summary: {old_summary or 'None'}\n\nNew messages:\n{convo}\n\nUpdated summary:"
    )
    return model.invoke(prompt).content

def chat_summary(user_text, keep_last=4):
    global summary, recent
    recent.append(HumanMessage(user_text))

    system_text = SYSTEM_TEXT
    if summary:
        system_text += f"\n\nSummary of the earlier conversation: {summary}"

    reply = model.invoke([SystemMessage(system_text)] + recent)
    recent.append(reply)

    # When recent gets too long, move the oldest messages into the summary
    if len(recent) > keep_last:
        old, recent = recent[:-keep_last], recent[-keep_last:]
        summary = update_summary(summary, old)

    print("User:", user_text)
    print("Bot :", reply.content)
    print(f"   [summary: {summary or '-'}]\n")

chat_summary("Hi, my name is Ravi.")
chat_summary("My order number is 1234 and it has not arrived.")
chat_summary("Do you deliver on Sundays?")
chat_summary("What is your return policy?")
chat_summary("What is my name and order number?")   # remembered via the summary

### Comparison

| Memory type | What is sent | Pros | Cons |
|---|---|---|---|
| **Buffer** | Everything | Perfect recall, simplest | Grows forever, so cost and limits become a problem |
| **Window** | Last *k* turns | Fixed, small cost | Forgets older facts |
| **Summary** | Summary + recent turns | Remembers key facts, low tokens | Extra LLM call; small details can be lost |

---
## Part 4: Token budgeting and context limits
Every model has a **context window**: a maximum number of tokens per request (input + output).
Plan it like a budget:

In [ ]:
CONTEXT_LIMIT = 8000   # example budget (real models allow more, but cost grows with every token)

budget = {
    "System prompt":            300,
    "Long-term customer facts": 200,
    "Conversation history":    5000,
    "Tool results / documents": 1500,
    "Reserved for the answer":  1000,
}
used = sum(budget.values())
for part, tokens in budget.items():
    print(f"{part:<26} {tokens:>5} tokens  {'#' * (tokens // 200)}")
print(f"{'TOTAL':<26} {used:>5} / {CONTEXT_LIMIT}  ->  {'OK' if used <= CONTEXT_LIMIT else 'OVER BUDGET!'}")

### Trimming: automatically fit the history into a token budget
`trim_messages` keeps the **system message** and the **most recent** messages that fit.

In [ ]:
from langchain_core.messages.utils import trim_messages

print("Before:", len(history), "messages,", count_tokens_approximately(history), "tokens")

trimmed = trim_messages(
    history,
    max_tokens=150,                      # our budget for history
    strategy="last",                     # keep the most recent messages
    token_counter=count_tokens_approximately,
    include_system=True,                 # always keep the system instructions
    start_on="human",                    # trimmed history should start with a user message
)

print("After :", len(trimmed), "messages,", count_tokens_approximately(trimmed), "tokens\n")
for m in trimmed:
    print(f"{m.type:>6}: {m.content[:70]}")

reply = model.invoke(trimmed + [HumanMessage("Summarize what we discussed in one line.")])
print("\nBot:", reply.content)

---
## Part 5: Passing context across steps (agents and tools)
A support agent often needs **several steps** for one question:

`User asks` → `LLM decides to call a tool` → `tool returns data (ToolMessage)` → `LLM uses that data` → `answer`

The output of one step becomes the **context for the next step**. Let's give our bot two tools.

In [ ]:
from langchain.tools import tool

# A tiny fake order database
ORDERS = {
    "1234": {"item": "Bluetooth headphones", "status": "Shipped",   "expected": "in 2 days"},
    "5678": {"item": "Phone cover",          "status": "Delivered", "expected": "-"},
}
TICKETS = []

@tool
def get_order_status(order_id: str) -> str:
    """Look up the status of a customer's order using its order ID, e.g. '1234'."""
    order = ORDERS.get(order_id.strip())
    if not order:
        return f"No order found with ID {order_id}."
    return f"Order {order_id}: {order['item']}, status: {order['status']}, expected delivery: {order['expected']}."

@tool
def raise_complaint(order_id: str, issue: str) -> str:
    """Raise a complaint ticket for an order. Needs the order ID and a short description of the issue."""
    ticket_id = f"TCK-{100 + len(TICKETS) + 1}"
    TICKETS.append({"ticket": ticket_id, "order_id": order_id, "issue": issue})
    return f"Complaint {ticket_id} raised for order {order_id}: {issue}"

tools = [get_order_status, raise_complaint]

# Test the tools directly (no LLM)
print(get_order_status.invoke({"order_id": "1234"}))

In [ ]:
from langchain.agents import create_agent

AGENT_PROMPT = (
    "You are ShopEasy's support assistant. Be short and friendly.\n"
    "Use get_order_status to check orders and raise_complaint to file complaints.\n"
    "If you don't know the order ID, ask the customer for it."
)

def ask_agent(agent, text, thread_id=None):
    """Send one message to the agent and print every step of this turn."""
    config = {"configurable": {"thread_id": thread_id}} if thread_id else {}
    result = agent.invoke({"messages": [HumanMessage(text)]}, config=config)

    msgs = result["messages"]
    start = max(i for i, m in enumerate(msgs) if m.type == "human")   # this turn's messages only
    print("User :", text)
    for m in msgs[start + 1:]:
        if m.type == "ai" and m.tool_calls:
            for tc in m.tool_calls:
                print(f"   step -> calls {tc['name']}({tc['args']})")
        elif m.type == "tool":
            print(f"   step <- tool result: {m.content}")
        elif m.type == "ai":
            print("Agent:", m.content)
    print()
    return result

# Agent WITHOUT memory
agent_no_memory = create_agent(model=model, tools=tools, system_prompt=AGENT_PROMPT)

ask_agent(agent_no_memory, "Hi, I'm Ravi. My order 1234 hasn't arrived. Where is it?")
ask_agent(agent_no_memory, "It's taking too long. Please raise a complaint.")

**Within one turn**, context passes between steps nicely (the tool result is used in the answer).

But **across turns** the agent forgets: on the second message it doesn't know the order ID and has to ask again. Customers hate repeating themselves!

---
## Part 6 (LAB): Add short-term memory to the agent
Two small changes:
1. A **checkpointer** (`InMemorySaver`) that saves the conversation after every turn
2. A **`thread_id`** that identifies the session (like a chat window)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

agent_with_memory = create_agent(
    model=model,
    tools=tools,
    system_prompt=AGENT_PROMPT,
    checkpointer=checkpointer,        # <- change 1: short-term memory
)

session = "ravi_session_1"            # <- change 2: one id per conversation

ask_agent(agent_with_memory, "Hi, I'm Ravi. My order 1234 hasn't arrived. Where is it?", thread_id=session)
ask_agent(agent_with_memory, "It's taking too long. Please raise a complaint.", thread_id=session)
ask_agent(agent_with_memory, "What's my complaint number, and what was my name?", thread_id=session)

In [ ]:
# What is stored for this session? (including tool calls and tool results)
config = {"configurable": {"thread_id": session}}
saved = agent_with_memory.get_state(config).values["messages"]

print(f"{len(saved)} messages saved, ~{count_tokens_approximately(saved)} tokens\n")
for m in saved:
    text = m.content if m.content else f"[tool call: {[tc['name'] for tc in m.tool_calls]}]"
    print(f"{m.type:>6}: {text[:80]}")

print("\nTickets in the system:", TICKETS)

✅ **The agent now remembers the order ID, the complaint and the customer name for the whole session.**

---
## Part 7: Session state vs longer-term state

| | **Short-term (session) memory** | **Long-term memory** |
|---|---|---|
| Lasts | One conversation (`thread_id`) | Across many sessions |
| Stores | Full message history | Key facts (name, preferences, last order) |
| How | Checkpointer (`InMemorySaver`, or a DB in production) | A store / database / file |
| Example | "Raise a complaint for *that* order" | "Welcome back Ravi! Is this about order 1234?" |

### A new session = a fresh start

In [ ]:
ask_agent(agent_with_memory, "What was my order number?", thread_id="ravi_session_2")

### Add simple long-term memory with a JSON file
We save a few important facts about the customer, and load them into the system prompt when a **new** session starts.

In [ ]:
import json

PROFILE_FILE = "customer_profiles.json"

def load_profiles():
    if os.path.exists(PROFILE_FILE):
        with open(PROFILE_FILE) as f:
            return json.load(f)
    return {}

def save_profile(customer_id, **facts):
    profiles = load_profiles()
    profiles.setdefault(customer_id, {}).update(facts)
    with open(PROFILE_FILE, "w") as f:
        json.dump(profiles, f, indent=2)
    print("Saved:", profiles[customer_id])

# At the end of session 1, save what matters (in real apps the agent or a background job does this)
save_profile("C001", name="Ravi", last_order="1234", open_complaint=TICKETS[-1]["ticket"] if TICKETS else None)

In [ ]:
def start_session(customer_id):
    """Create an agent for a new session, pre-loaded with the customer's long-term facts."""
    profile = load_profiles().get(customer_id, {})
    prompt = AGENT_PROMPT
    if profile:
        prompt += f"\nKnown facts about this customer from earlier sessions: {json.dumps(profile)}"
    return create_agent(model=model, tools=tools, system_prompt=prompt, checkpointer=checkpointer)

# Days later: a brand-new session (new thread_id), but the bot still knows Ravi
agent_session3 = start_session("C001")
ask_agent(agent_session3, "Hi again! Any update on my last order?", thread_id="ravi_session_3")

---
## Practice exercises
1. In Part 3a, change `k=1` to `k=3`. Does the bot remember the name now? How many tokens are sent?
2. In Part 3b, change `keep_last=4` to `keep_last=2`. What happens to the summary?
3. In Part 4, set `max_tokens=60`. Which messages survive the trim?
4. Add a new tool `cancel_order(order_id)` and test: *"Check order 5678… actually, cancel it."*
5. **Challenge:** add a tool `remember_fact(key, value)` that calls `save_profile`, so the agent decides by itself what to remember long-term.

In [ ]:
# Interactive chat with the memory agent (type 'exit' to stop). Uncomment to try.
# while True:
#     text = input("You: ")
#     if text.strip().lower() in {"exit", "quit"}:
#         break
#     ask_agent(agent_with_memory, text, thread_id="my_live_session")

## Summary
- LLMs are **stateless**: memory = **re-sending context** on every call.
- **Buffer** keeps all (simple, grows), **window** keeps the last k (cheap, forgets), **summary** compresses old turns (balanced).
- Plan a **token budget**; use `trim_messages` to stay inside the context limit.
- In agents, each step's output (tool results) becomes context for the next step.
- **Short-term memory** = checkpointer + `thread_id` (one session). **Long-term memory** = facts saved in a store and loaded into new sessions.